# Предобработка данных

На этом этапе проводится первичный анализ и подготовка исходного датасета к обучению моделей.

Основные задачи:
- проверить структуру и качество данных;
- определить целевые и служебные признаки;
- исследовать пропуски и типы данных;
- исключить возможную утечку целевой информации;
- сформировать итоговую матрицу признаков `X`.


## 1. Загрузка данных

Загружаем исходный датасет из `data/raw` и проверяем его размер и первые строки. Исходный файл на этом этапе не изменяется.


In [3]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/deficiency_anemia.csv"

df = pd.read_csv(DATA_PATH)

print("Размер датасета:", df.shape)
print("\nПервые 5 строк:")
display(df.head())

Размер датасета: (840, 48)

Первые 5 строк:


,patient_id,age_years,sex,hemoglobin,RBC,hematocrit,MCV,MCH,MCHC,RDW,...,anemia,iron_deficiency,B12_deficiency,folate_deficiency,B6_deficiency,copper_deficiency,inflammation_anemia,mixed_deficiency,anemia_class,deficiency_cause
0,P000001,64,M,139.6,4.99,41.8,83.3,28.0,332.5,13.3,...,0,0,1,0,0,0,0,0,B12_deficiency_no_anemia,B12_deficiency
1,P000002,52,F,122.3,3.87,35.5,90.0,31.6,343.7,19.8,...,0,0,0,1,0,0,0,0,folate_deficiency_no_anemia,folate_deficiency
2,P000003,43,F,144.9,5.20,44.6,86.1,27.9,329.2,14.7,...,0,1,0,0,0,0,0,0,latent_deficiency,iron_deficiency
3,P000004,26,F,121.7,4.46,39.0,86.9,27.3,316.2,16.7,...,0,0,1,0,0,0,0,0,B12_deficiency_no_anemia,B12_deficiency
4,P000005,30,F,111.5,2.98,32.5,109.1,37.5,341.6,19.7,...,1,0,1,0,0,0,0,0,B12_deficiency_anemia,B12_deficiency


## 2. Проверка структуры датасета

Изучаем типы данных, количество заполненных значений и названия столбцов. Это помогает понять структуру исходных данных перед дальнейшей обработкой.


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 48 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   patient_id           840 non-null    str    
 1   age_years            840 non-null    int64  
 2   sex                  840 non-null    str    
 3   hemoglobin           840 non-null    float64
 4   RBC                  840 non-null    float64
 5   hematocrit           840 non-null    float64
 6   MCV                  840 non-null    float64
 7   MCH                  840 non-null    float64
 8   MCHC                 840 non-null    float64
 9   RDW                  825 non-null    float64
 10  platelets            840 non-null    int64  
 11  WBC                  840 non-null    float64
 12  reticulocytes        401 non-null    float64
 13  ferritin             594 non-null    float64
 14  serum_iron           540 non-null    float64
 15  transferrin          540 non-null    float64
 16  T

In [5]:
df.columns.tolist()

['patient_id',
 'age_years',
 'sex',
 'hemoglobin',
 'RBC',
 'hematocrit',
 'MCV',
 'MCH',
 'MCHC',
 'RDW',
 'platelets',
 'WBC',
 'reticulocytes',
 'ferritin',
 'serum_iron',
 'transferrin',
 'TIBC',
 'UIBC',
 'TSAT',
 'sTfR',
 'Ret_He',
 'vitamin_B12',
 'active_B12',
 'MMA',
 'homocysteine',
 'folate',
 'vitamin_B6',
 'copper',
 'ceruloplasmin',
 'CRP',
 'ESR',
 'creatinine',
 'eGFR',
 'TSH',
 'albumin',
 'LDH',
 'indirect_bilirubin',
 'haptoglobin',
 'anemia',
 'iron_deficiency',
 'B12_deficiency',
 'folate_deficiency',
 'B6_deficiency',
 'copper_deficiency',
 'inflammation_anemia',
 'mixed_deficiency',
 'anemia_class',
 'deficiency_cause']

## 3. Проверка пациентов

Проверяем количество уникальных пациентов и наличие повторяющихся `patient_id`. Это важно для корректного последующего разделения данных.


In [6]:
df["patient_id"].nunique()

840

In [7]:
df["patient_id"].duplicated().sum()

np.int64(0)

## 4. Анализ целевого класса

Проверяем распределение и количество значений `anemia_class`. Этот признак будет использоваться как основная целевая переменная многоклассовой классификации.


In [8]:
df["anemia_class"].value_counts(dropna=False)

anemia_class
iron_deficiency_anemia         115
mixed_deficiency               115
no_anemia_no_deficiency        110
latent_deficiency               85
anemia_other                    75
B12_deficiency_anemia           65
inflammation_anemia             65
B12_deficiency_no_anemia        50
folate_deficiency_anemia        50
folate_deficiency_no_anemia     40
copper_deficiency               35
B6_deficiency                   35
Name: count, dtype: int64

In [9]:
df["anemia_class"].nunique()

12

## 5. Целевые и идентификационные колонки

Отдельно фиксируем целевые признаки и идентификатор пациента. Эти столбцы не должны попадать в матрицу входных признаков модели, чтобы избежать утечки целевой информации (`data leakage`).


In [10]:
TARGET_COLUMNS = [
    "anemia",
    "iron_deficiency",
    "B12_deficiency",
    "folate_deficiency",
    "B6_deficiency",
    "copper_deficiency",
    "inflammation_anemia",
    "mixed_deficiency",
    "anemia_class",
    "deficiency_cause",
]

ID_COLUMNS = [
    "patient_id",
]

## 6. Первичная проверка типов и пропусков

Смотрим типы всех колонок и долю отсутствующих значений. На этом этапе пропуски не заполняются и строки не удаляются.


In [11]:
df.dtypes

patient_id                 str
age_years                int64
sex                        str
hemoglobin             float64
RBC                    float64
hematocrit             float64
MCV                    float64
MCH                    float64
MCHC                   float64
RDW                    float64
platelets                int64
WBC                    float64
reticulocytes          float64
ferritin               float64
serum_iron             float64
transferrin            float64
TIBC                   float64
UIBC                   float64
TSAT                   float64
sTfR                   float64
Ret_He                 float64
vitamin_B12            float64
active_B12             float64
MMA                    float64
homocysteine           float64
folate                 float64
vitamin_B6             float64
copper                 float64
ceruloplasmin          float64
CRP                    float64
ESR                    float64
creatinine             float64
eGFR    

In [12]:
missing_percent = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing_percent

copper                 85.238095
ceruloplasmin          84.404762
vitamin_B6             83.928571
Ret_He                 73.095238
sTfR                   70.000000
active_B12             70.000000
MMA                    67.619048
homocysteine           64.523810
LDH                    61.309524
indirect_bilirubin     61.309524
haptoglobin            58.809524
reticulocytes          52.261905
TSH                    49.404762
folate                 45.833333
ESR                    41.428571
CRP                    40.952381
vitamin_B12            38.928571
TIBC                   35.714286
UIBC                   35.714286
TSAT                   35.714286
serum_iron             35.714286
transferrin            35.714286
eGFR                   35.595238
creatinine             35.595238
albumin                35.238095
ferritin               29.285714
RDW                     1.785714
patient_id              0.000000
age_years               0.000000
sex                     0.000000
MCV       

## 7. Проверка целевых признаков

Убеждаемся, что все ожидаемые целевые и идентификационные колонки присутствуют в датасете, после чего рассматриваем реальные значения целевых переменных.


In [13]:
missing_targets = [
    col for col in TARGET_COLUMNS
    if col not in df.columns
]

missing_ids = [
    col for col in ID_COLUMNS
    if col not in df.columns
]

print("Отсутствующие target-колонки:", missing_targets)
print("Отсутствующие ID-колонки:", missing_ids)

Отсутствующие target-колонки: []
Отсутствующие ID-колонки: []


In [14]:
for col in TARGET_COLUMNS:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- anemia ---
anemia
1    531
0    309
Name: count, dtype: int64

--- iron_deficiency ---
iron_deficiency
0    550
1    290
Name: count, dtype: int64

--- B12_deficiency ---
B12_deficiency
0    650
1    190
Name: count, dtype: int64

--- folate_deficiency ---
folate_deficiency
0    685
1    155
Name: count, dtype: int64

--- B6_deficiency ---
B6_deficiency
0    805
1     35
Name: count, dtype: int64

--- copper_deficiency ---
copper_deficiency
0    805
1     35
Name: count, dtype: int64

--- inflammation_anemia ---
inflammation_anemia
0    775
1     65
Name: count, dtype: int64

--- mixed_deficiency ---
mixed_deficiency
0    725
1    115
Name: count, dtype: int64

--- anemia_class ---
anemia_class
iron_deficiency_anemia         115
mixed_deficiency               115
no_anemia_no_deficiency        110
latent_deficiency               85
anemia_other                    75
B12_deficiency_anemia           65
inflammation_anemia             65
B12_deficiency_no_anemia        50
folate_defi

## 8. Проверка дубликатов

Проверяем наличие полностью одинаковых строк. Найденные дубликаты не удаляются автоматически: сначала необходимо определить причину их появления.


In [15]:
n_full_duplicates = df.duplicated().sum()

print("Полностью одинаковых строк:", n_full_duplicates)

Полностью одинаковых строк: 0


## 9. Анализ строковых признаков

Определяем колонки строкового типа и рассматриваем содержащиеся в них значения. Это позволяет отделить категориальные признаки от служебных полей и других типов данных.


In [16]:
object_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Строковые колонки:")
for col in object_columns:
    print(col)

Строковые колонки:
patient_id
sex
anemia_class
deficiency_cause


/tmp/ipykernel_20942/4277390358.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include=["object"]).columns.tolist()


In [17]:
for col in object_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(30))


--- patient_id ---
patient_id
P000001    1
P000002    1
P000003    1
P000004    1
P000005    1
P000006    1
P000007    1
P000008    1
P000009    1
P000010    1
P000011    1
P000012    1
P000013    1
P000014    1
P000015    1
P000016    1
P000017    1
P000018    1
P000019    1
P000020    1
P000021    1
P000022    1
P000023    1
P000024    1
P000025    1
P000026    1
P000027    1
P000028    1
P000029    1
P000030    1
Name: count, dtype: int64

--- sex ---
sex
F    596
M    244
Name: count, dtype: int64

--- anemia_class ---
anemia_class
iron_deficiency_anemia         115
mixed_deficiency               115
no_anemia_no_deficiency        110
latent_deficiency               85
anemia_other                    75
B12_deficiency_anemia           65
inflammation_anemia             65
B12_deficiency_no_anemia        50
folate_deficiency_anemia        50
folate_deficiency_no_anemia     40
copper_deficiency               35
B6_deficiency                   35
Name: count, dtype: int64

--- defici

## 10. Проверка числовых значений

Проверяем числовые признаки на наличие `inf` и `-inf`. Такие значения нельзя напрямую использовать при обучении большинства моделей.


In [18]:
numeric_columns = df.select_dtypes(include=np.number).columns

inf_counts = pd.Series({
    col: np.isinf(df[col]).sum()
    for col in numeric_columns
})

inf_counts = inf_counts[inf_counts > 0]

print("Колонки с inf/-inf:")
display(inf_counts)

Колонки с inf/-inf:


Series([], dtype: int64)

## 11. Анализ пропущенных значений

Формируем таблицу с количеством и процентом пропусков по каждому признаку. Пропуски пока сохраняются в исходном виде и не заменяются искусственными значениями.


In [19]:
missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing_table = (
    missing_table
    .query("missing_count > 0")
    .sort_values("missing_percent", ascending=False)
)

display(missing_table)

,missing_count,missing_percent
copper,716,85.238095
ceruloplasmin,709,84.404762
vitamin_B6,705,83.928571
Ret_He,614,73.095238
active_B12,588,70.000000
sTfR,588,70.000000
MMA,568,67.619048
homocysteine,542,64.523810
LDH,515,61.309524
indirect_bilirubin,515,61.309524


## 12. Пропуски в целевых переменных

Отдельно проверяем пропуски в target-колонках. Отсутствующее целевое значение нельзя автоматически интерпретировать как отрицательный класс.


In [20]:
target_missing = df[TARGET_COLUMNS].isna().sum()

print("Пропуски в target-колонках:")
display(target_missing)

Пропуски в target-колонках:


anemia                 0
iron_deficiency        0
B12_deficiency         0
folate_deficiency      0
B6_deficiency          0
copper_deficiency      0
inflammation_anemia    0
mixed_deficiency       0
anemia_class           0
deficiency_cause       0
dtype: int64

## 13. Анализ потенциальных признаков модели

Отделяем потенциальные входные признаки от targets и идентификатора пациента и рассматриваем основные статистики числовых показателей.


In [21]:
feature_candidates = [
    col for col in df.columns
    if col not in TARGET_COLUMNS + ID_COLUMNS
]

numeric_feature_candidates = [
    col for col in feature_candidates
    if pd.api.types.is_numeric_dtype(df[col])
]

print("Количество потенциальных признаков:", len(feature_candidates))
print("Из них числовых:", len(numeric_feature_candidates))

Количество потенциальных признаков: 37
Из них числовых: 36


In [22]:
display(
    df[numeric_feature_candidates]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
age_years,840.0,58.405952,19.070991,18.000,44.0000,60.000,74.0000,93.00
hemoglobin,840.0,114.264167,22.294276,52.300,96.5750,111.700,133.7250,174.80
RBC,840.0,4.013488,0.880699,2.020,3.3575,4.000,4.6500,6.68
hematocrit,840.0,35.456071,6.600880,17.900,30.6000,35.100,40.7250,53.60
MCV,840.0,89.621667,10.708688,58.000,83.2750,88.400,94.9250,123.40
MCH,840.0,28.902024,3.925851,16.700,26.4000,28.700,31.2000,40.70
MCHC,840.0,322.089167,13.611821,270.000,314.3000,322.600,331.1000,359.90
RDW,825.0,16.175515,3.027909,11.100,13.9000,15.500,17.8000,30.20
platelets,840.0,273.698810,88.781600,76.000,213.7500,266.500,324.0000,722.00
WBC,840.0,6.163893,2.003560,1.900,4.7575,5.895,7.3600,14.34


## 14. Проверка информативности признаков

Ищем константные признаки и признаки с очень большой долей пропусков. Они рассматриваются как кандидаты на исключение, но на этом этапе автоматически не удаляются.


In [23]:
constant_columns = [
    col for col in feature_candidates
    if df[col].nunique(dropna=False) <= 1
]

print("Константные признаки:")
print(constant_columns)

Константные признаки:
[]


In [24]:
high_missing_columns = (
    df[feature_candidates]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

high_missing_columns = high_missing_columns[
    high_missing_columns > 0.80
]

print("Признаки с >80% пропусков:")
display(high_missing_columns)

Признаки с >80% пропусков:


copper           0.852381
ceruloplasmin    0.844048
vitamin_B6       0.839286
dtype: float64

## 15. Формирование `X` и `y`

Создаём матрицу входных признаков `X` и целевой вектор `y`. Из `X` исключаются все target-колонки и `patient_id`, чтобы модель не получила правильный ответ непосредственно во входных данных.


In [25]:
DROP_COLUMNS = TARGET_COLUMNS + ID_COLUMNS

X = df.drop(
    columns=DROP_COLUMNS,
    errors="raise"
).copy()

y = df["anemia_class"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

X shape: (840, 37)
y shape: (840,)


,age_years,sex,hemoglobin,RBC,hematocrit,MCV,MCH,MCHC,RDW,platelets,...,ceruloplasmin,CRP,ESR,creatinine,eGFR,TSH,albumin,LDH,indirect_bilirubin,haptoglobin
0,64,M,139.6,4.99,41.8,83.3,28.0,332.5,13.3,297,...,NaN,1.5,19.0,85.0,78.0,NaN,NaN,150.0,NaN,NaN
1,52,F,122.3,3.87,35.5,90.0,31.6,343.7,19.8,234,...,NaN,1.2,NaN,65.0,114.0,8.46,NaN,NaN,NaN,NaN
2,43,F,144.9,5.20,44.6,86.1,27.9,329.2,14.7,288,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.87
3,26,F,121.7,4.46,39.0,86.9,27.3,316.2,16.7,386,...,NaN,NaN,3.0,66.0,110.0,1.87,NaN,NaN,NaN,NaN
4,30,F,111.5,2.98,32.5,109.1,37.5,341.6,19.7,281,...,NaN,1.0,NaN,78.0,108.0,2.54,43.2,500.0,29.9,NaN


## 16. Проверка на data leakage

Дополнительно убеждаемся, что в `X` действительно не осталось целевых или идентификационных колонок.


In [26]:
leakage_columns = [
    col for col in TARGET_COLUMNS + ID_COLUMNS
    if col in X.columns
]

print("Leakage columns в X:", leakage_columns)

assert len(leakage_columns) == 0, (
    f"В X обнаружены запрещённые колонки: {leakage_columns}"
)

print("Проверка пройдена: targets и patient_id отсутствуют в X.")

Leakage columns в X: []
Проверка пройдена: targets и patient_id отсутствуют в X.


## 17. Числовые и категориальные признаки

Разделяем оставшиеся признаки по типам. Это потребуется позже, поскольку числовые и категориальные данные могут обрабатываться моделью по-разному.


In [27]:
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

print("Числовых признаков:", len(numeric_features))
print("Категориальных признаков:", len(categorical_features))

print("\nКатегориальные:")
print(categorical_features)

Числовых признаков: 36
Категориальных признаков: 1

Категориальные:
['sex']


/tmp/ipykernel_20942/1373298514.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


## 18. Нормализация пола

Проверяем реальные значения признака `sex`, после чего приводим строковые значения к единому регистру и удаляем лишние пробелы.


In [28]:
if "sex" in X.columns:
    print(X["sex"].value_counts(dropna=False))

sex
F    596
M    244
Name: count, dtype: int64


In [29]:
X["sex"] = (
    X["sex"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(X["sex"].value_counts(dropna=False))

sex
f    596
m    244
Name: count, dtype: Int64


## 19. Проверка служебных признаков

Ищем поля, которые могут описывать происхождение или технические характеристики записи, а не состояние пациента. Такие признаки отдельно проверяются на возможную утечку информации.


In [30]:
possible_meta = [
    "record_origin",
    "birth_date",
    "anchor_sample_date",
]

existing_meta = [
    col for col in possible_meta
    if col in X.columns
]

print("Найденные потенциально служебные признаки:")
print(existing_meta)

Найденные потенциально служебные признаки:
[]


In [31]:
if "record_origin" in df.columns:
    display(
        pd.crosstab(
            df["record_origin"],
            df["anemia_class"],
            normalize="index"
        ).round(3)
    )

## 20. Финальный набор признаков

Исключаем выбранные служебные поля и фиксируем окончательный список `FEATURE_COLUMNS`. Один и тот же набор и порядок признаков должен использоваться при обучении и последующем inference.


In [32]:
META_COLUMNS = [
    col for col in [
        "record_origin",
        "birth_date",
        "anchor_sample_date",
    ]
    if col in X.columns
]

X = X.drop(
    columns=META_COLUMNS,
    errors="ignore"
).copy()

FEATURE_COLUMNS = X.columns.tolist()

print("Количество финальных признаков:", len(FEATURE_COLUMNS))

for i, col in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{i:2}. {col}")

Количество финальных признаков: 37
 1. age_years
 2. sex
 3. hemoglobin
 4. RBC
 5. hematocrit
 6. MCV
 7. MCH
 8. MCHC
 9. RDW
10. platelets
11. WBC
12. reticulocytes
13. ferritin
14. serum_iron
15. transferrin
16. TIBC
17. UIBC
18. TSAT
19. sTfR
20. Ret_He
21. vitamin_B12
22. active_B12
23. MMA
24. homocysteine
25. folate
26. vitamin_B6
27. copper
28. ceruloplasmin
29. CRP
30. ESR
31. creatinine
32. eGFR
33. TSH
34. albumin
35. LDH
36. indirect_bilirubin
37. haptoglobin


## 21. Маска пропусков и полнота данных

Сохраняем информацию о том, какие показатели действительно присутствуют у каждого пациента, и рассчитываем долю доступных признаков (`feature coverage`). Пока эта величина используется только для анализа качества входных данных.


In [33]:
missing_mask = X.isna()

display(missing_mask.head())

,age_years,sex,hemoglobin,RBC,hematocrit,MCV,MCH,MCHC,RDW,platelets,...,ceruloplasmin,CRP,ESR,creatinine,eGFR,TSH,albumin,LDH,indirect_bilirubin,haptoglobin
0,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,True,True,False,True,True
1,False,False,False,False,False,False,False,False,False,False,...,True,False,True,False,False,False,True,True,True,True
2,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,False
3,False,False,False,False,False,False,False,False,False,False,...,True,True,False,False,False,False,True,True,True,True
4,False,False,False,False,False,False,False,False,False,False,...,True,False,True,False,False,False,False,False,False,True


In [34]:
feature_coverage = X.notna().mean(axis=1)

print(feature_coverage.describe())

count    840.000000
mean       0.631049
std        0.103973
min        0.324324
25%        0.567568
50%        0.648649
75%        0.702703
max        0.891892
dtype: float64


In [35]:
coverage_table = pd.DataFrame({
    "patient_id": df["patient_id"],
    "feature_coverage": feature_coverage
})

display(
    coverage_table
    .sort_values("feature_coverage")
    .head(20)
)

,patient_id,feature_coverage
699,P000700,0.324324
388,P000389,0.351351
89,P000090,0.351351
109,P000110,0.378378
242,P000243,0.378378
713,P000714,0.405405
356,P000357,0.405405
418,P000419,0.405405
85,P000086,0.405405
108,P000109,0.405405


## 22. Распределение классов перед разделением данных

Перед созданием обучающей, валидационной и тестовой выборок ещё раз оцениваем баланс классов. Это понадобится для корректного стратифицированного разделения.


In [36]:
class_distribution = pd.DataFrame({
    "count": y.value_counts(),
    "percent": y.value_counts(normalize=True) * 100
})

display(class_distribution)

,count,percent
anemia_class,,
iron_deficiency_anemia,115,13.690476
mixed_deficiency,115,13.690476
no_anemia_no_deficiency,110,13.095238
latent_deficiency,85,10.119048
anemia_other,75,8.928571
B12_deficiency_anemia,65,7.738095
inflammation_anemia,65,7.738095
B12_deficiency_no_anemia,50,5.952381
folate_deficiency_anemia,50,5.952381


## 23. Следующий этап: разделение данных

Следующий шаг — разделить пациентов на обучающую, валидационную и тестовую выборки.

Разделение необходимо выполнить **до** обучения imputer, scaler и модели, чтобы информация из validation и test не влияла на процесс обучения.

На этом этапе preprocessing пока остановлен. Далее отдельно разберём логику `train / validation / test`, стратификацию и проверку отсутствия пересечений пациентов между выборками.


In [37]:
from sklearn.model_selection import train_test_split

In [38]:
train_idx, temp_idx = train_test_split(
    df.index,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Train:", len(train_idx))
print("Temp:", len(temp_idx))

Train: 588
Temp: 252


In [39]:
valid_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=y.loc[temp_idx]
)

print("Train:", len(train_idx))
print("Validation:", len(valid_idx))
print("Test:", len(test_idx))

Train: 588
Validation: 126
Test: 126


### Формирование выборок

По полученным индексам создаём отдельные матрицы признаков и целевые переменные для train, validation и test.

In [40]:
X_train = X.loc[train_idx].copy()
X_valid = X.loc[valid_idx].copy()
X_test = X.loc[test_idx].copy()

y_train = y.loc[train_idx].copy()
y_valid = y.loc[valid_idx].copy()
y_test = y.loc[test_idx].copy()

print("Train:")
print("X:", X_train.shape, "y:", y_train.shape)

print("\nValidation:")
print("X:", X_valid.shape, "y:", y_valid.shape)

print("\nTest:")
print("X:", X_test.shape, "y:", y_test.shape)

Train:
X: (588, 37) y: (588,)

Validation:
X: (126, 37) y: (126,)

Test:
X: (126, 37) y: (126,)


### Проверка независимости выборок

Проверяем, что один и тот же пациент не попал одновременно в несколько выборок. Это предотвращает утечку информации между train, validation и test.

In [41]:
train_ids = set(df.loc[train_idx, "patient_id"])
valid_ids = set(df.loc[valid_idx, "patient_id"])
test_ids = set(df.loc[test_idx, "patient_id"])

print(
    "Train ∩ Validation:",
    len(train_ids & valid_ids)
)

print(
    "Train ∩ Test:",
    len(train_ids & test_ids)
)

print(
    "Validation ∩ Test:",
    len(valid_ids & test_ids)
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [42]:
assert train_ids.isdisjoint(valid_ids)
assert train_ids.isdisjoint(test_ids)
assert valid_ids.isdisjoint(test_ids)

print("✓ Пересечений пациентов между выборками нет.")

✓ Пересечений пациентов между выборками нет.


In [43]:
total = len(df)

print(f"Train:      {len(train_idx):4d} ({len(train_idx) / total:.1%})")
print(f"Validation: {len(valid_idx):4d} ({len(valid_idx) / total:.1%})")
print(f"Test:       {len(test_idx):4d} ({len(test_idx) / total:.1%})")

Train:       588 (70.0%)
Validation:  126 (15.0%)
Test:        126 (15.0%)


### Проверка распределения классов после разделения

После стратифицированного разделения проверяем распределение `anemia_class` в train, validation и test.

Доли классов должны быть примерно одинаковыми во всех трёх выборках. Это особенно важно при наличии редких классов.

In [44]:
def class_distribution(target):
    return pd.DataFrame({
        "count": target.value_counts(),
        "percent": (
            target.value_counts(normalize=True) * 100
        ).round(2)
    })


print("TRAIN")
display(class_distribution(y_train))

print("VALIDATION")
display(class_distribution(y_valid))

print("TEST")
display(class_distribution(y_test))

TRAIN


,count,percent
anemia_class,,
mixed_deficiency,81,13.78
iron_deficiency_anemia,81,13.78
no_anemia_no_deficiency,77,13.10
latent_deficiency,59,10.03
anemia_other,52,8.84
B12_deficiency_anemia,46,7.82
inflammation_anemia,45,7.65
folate_deficiency_anemia,35,5.95
B12_deficiency_no_anemia,35,5.95


VALIDATION


,count,percent
anemia_class,,
no_anemia_no_deficiency,17,13.49
mixed_deficiency,17,13.49
iron_deficiency_anemia,17,13.49
latent_deficiency,13,10.32
anemia_other,11,8.73
inflammation_anemia,10,7.94
B12_deficiency_anemia,10,7.94
B12_deficiency_no_anemia,8,6.35
folate_deficiency_anemia,7,5.56


TEST


,count,percent
anemia_class,,
iron_deficiency_anemia,17,13.49
mixed_deficiency,17,13.49
no_anemia_no_deficiency,16,12.70
latent_deficiency,13,10.32
anemia_other,12,9.52
inflammation_anemia,10,7.94
B12_deficiency_anemia,9,7.14
folate_deficiency_anemia,8,6.35
B12_deficiency_no_anemia,7,5.56


In [45]:
distribution_comparison = pd.DataFrame({
    "full_%": y.value_counts(normalize=True) * 100,
    "train_%": y_train.value_counts(normalize=True) * 100,
    "valid_%": y_valid.value_counts(normalize=True) * 100,
    "test_%": y_test.value_counts(normalize=True) * 100,
}).round(2)

display(distribution_comparison)

,full_%,train_%,valid_%,test_%
anemia_class,,,,
B12_deficiency_anemia,7.74,7.82,7.94,7.14
B12_deficiency_no_anemia,5.95,5.95,6.35,5.56
B6_deficiency,4.17,4.25,3.97,3.97
anemia_other,8.93,8.84,8.73,9.52
copper_deficiency,4.17,4.08,3.97,4.76
folate_deficiency_anemia,5.95,5.95,5.56,6.35
folate_deficiency_no_anemia,4.76,4.76,4.76,4.76
inflammation_anemia,7.74,7.65,7.94,7.94
iron_deficiency_anemia,13.69,13.78,13.49,13.49


In [46]:
all_classes = set(y.unique())

print("Количество классов во всём датасете:", len(all_classes))
print("Train:", len(set(y_train.unique())))
print("Validation:", len(set(y_valid.unique())))
print("Test:", len(set(y_test.unique())))

print("\nОтсутствуют в train:",
      all_classes - set(y_train.unique()))

print("Отсутствуют в validation:",
      all_classes - set(y_valid.unique()))

print("Отсутствуют в test:",
      all_classes - set(y_test.unique()))

Количество классов во всём датасете: 12
Train: 12
Validation: 12
Test: 12

Отсутствуют в train: set()
Отсутствуют в validation: set()
Отсутствуют в test: set()


## 25. Подготовка preprocessing для моделей

В соответствии с планом сравниваются три модели:

- Logistic Regression;
- Random Forest;
- CatBoost.

Все модели обучаются на одинаковом разделении данных, однако preprocessing для них различается.

Параметры preprocessing обучаются только на тренировочных данных. Validation и test используются только для применения уже обученных преобразований и оценки моделей.

In [47]:
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Всего признаков:", X_train.shape[1])
print("Числовых признаков:", len(numeric_features))
print("Категориальных признаков:", len(categorical_features))

print("\nКатегориальные признаки:")
print(categorical_features)

Всего признаков: 37
Числовых признаков: 36
Категориальных признаков: 1

Категориальные признаки:
['sex']


### Анализ пропусков в обучающей выборке

После разделения данных повторно анализируем пропуски в `X_train`.

Стратегия заполнения будет обучаться только на тренировочной выборке, чтобы информация из validation и test не использовалась при подготовке модели.

In [48]:
train_missing = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_percent": X_train.isna().mean() * 100
})

train_missing = (
    train_missing
    .query("missing_count > 0")
    .sort_values("missing_percent", ascending=False)
)

display(train_missing)

,missing_count,missing_percent
copper,502,85.374150
vitamin_B6,498,84.693878
ceruloplasmin,493,83.843537
Ret_He,436,74.149660
sTfR,420,71.428571
active_B12,410,69.727891
MMA,399,67.857143
homocysteine,373,63.435374
LDH,364,61.904762
indirect_bilirubin,364,61.904762


In [49]:
for col in categorical_features:
    print(f"\n--- {col} ---")
    print(X_train[col].value_counts(dropna=False))


--- sex ---
sex
f    411
m    177
Name: count, dtype: Int64


## 27. Preprocessing для Logistic Regression

Для Logistic Regression необходимо подготовить числовые и категориальные признаки.

Числовые признаки:
- пропущенные значения заполняются медианой, рассчитанной только на train;
- после заполнения признаки стандартизируются с помощью `StandardScaler`.

Категориальный признак `sex`:
- обрабатывается отдельно;
- преобразуется в числовое представление с помощью `OneHotEncoder`.

Все параметры preprocessing обучаются только на тренировочной выборке.

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [52]:
numeric_lr_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

In [53]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="if_binary"
        )
    )
])

In [54]:
lr_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_lr_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

## 28. Preprocessing для Random Forest

Random Forest также требует обработки пропусков и категориальных признаков.

В отличие от Logistic Regression, масштабирование числовых признаков для Random Forest не требуется, поскольку модель основана на деревьях решений.

In [55]:
numeric_rf_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

In [56]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_rf_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [57]:
print("Logistic Regression preprocessor:")
print(lr_preprocessor)

print("\nRandom Forest preprocessor:")
print(rf_preprocessor)

Logistic Regression preprocessor:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age_years', 'hemoglobin', 'RBC',
                                  'hematocrit', 'MCV', 'MCH', 'MCHC', 'RDW',
                                  'platelets', 'WBC', 'reticulocytes',
                                  'ferritin', 'serum_iron', 'transferrin',
                                  'TIBC', 'UIBC', 'TSAT', 'sTfR', 'Ret_He',
                                  'vitamin_B12', 'active_B12', 'MMA',
                                  'homocysteine', 'folate', 'vitamin_B6',
                                  'copper', 'ceruloplasmin', 'CRP', 'ESR',
                                  'creatinine', ...]),
                                ('cat',
                  